# Notebook 17v2 — Verification Battery
## Is MELU-Δt 99.8% on Arrhythmia genuine?

When a result in the literature is unusually high (99.8% vs published best 67.2%),
five diagnostic tests determine whether it is real:

| Test | Question | Method |
|---|---|---|
| 1 — ELU-AE | Does MELU specifically improve on plain ELU? | Same AE, ELU activation |
| 2 — Shallow | Is ANY one-class method near-perfect? | IForest, LOF, OCSVM |
| 3 — Random AE | Does an untrained AE already separate classes? | Random weights |
| 4 — Score plot | Is there full separation or overlap? | Histogram |
| 5 — Supervised | Are classes linearly separable at all? | Logistic regression |

## Decision tree

```
Random AE AUROC > 0.80 → TRIVIAL: feature structure alone separates, any method wins
Random AE AUROC ~ 0.50 ↓
  IForest AUROC > 0.95  → EASY: dataset not suitable for deep-method benchmark
  IForest AUROC < 0.85  ↓
    ELU-AE F1 ~ MELU F1 → AE-DRIVEN: reconstruction error beats energy/transform methods,
                           but MELU adds nothing over ELU
    ELU-AE << MELU        → GENUINE: activation function makes the difference ← desired result
```

A supervised LR upper bound confirms whether the classes are fundamentally separable.
If LR F1 < 70%, the dataset is genuinely hard and 99.8% is suspicious.
If LR F1 > 95%, the classes ARE separable and high detection rates are plausible.


## Cell 1 — Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import betainc, gammaln
from scipy.io import loadmat, savemat
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression
import warnings, time, os, urllib.request
warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)

# ── Protocol ──────────────────────────────────────────────────────────────────
N_RUNS       = 20
TRAIN_FRAC   = 0.50
LR           = 1e-3
EPOCHS_PRE   = 60
EPOCHS_FINE  = 80
LAM_BCE      = 0.5
PCT_PSEUDO   = 85
TAU_PERCENTILE = 75

def lat_for(dim): return max(4, min(dim // 2, 16))
def hid_for(dim): return max(64, dim * 4)

print("All imports OK")
print()
print("This notebook runs 5 diagnostic tests on each dataset.")
print("Purpose: verify whether MELU-Dt 99.8% F1 on Arrhythmia is genuine.")


All imports OK

This notebook runs 5 diagnostic tests on each dataset.
Purpose: verify whether MELU-Dt 99.8% F1 on Arrhythmia is genuine.


## Cell 2 — Dataset loading

In [2]:
os.makedirs("data", exist_ok=True)

def try_pyod(name, path):
    try:
        from pyod.utils.data import load_mat_file
        X, y = load_mat_file(name)
        savemat(path, {"X": X, "y": y.reshape(-1,1)}); return True
    except: return False

def try_url(name, path):
    urls = {"arrhythmia":"http://odds.cs.stonybrook.edu/static/media/releases/datasets/arrhythmia.mat",
            "thyroid":"http://odds.cs.stonybrook.edu/static/media/releases/datasets/thyroid.mat"}
    try: urllib.request.urlretrieve(urls[name], path); return True
    except: return False

def simulate(name, path):
    stats = {"arrhythmia":(386,66,274,0.15), "thyroid":(7018,182,6,0.50)}
    n_in,n_out,dim,rho = stats[name]
    print(f"  [SIM] WARNING: {name} is SIMULATED — not for paper comparison")
    np.random.seed(42)
    cov = np.array([[rho**abs(i-j) for j in range(dim)] for i in range(dim)]).astype(np.float32)
    cov += np.eye(dim,dtype=np.float32)*0.05; L=np.linalg.cholesky(cov)
    Xi = (np.random.randn(n_in,dim)@L.T).astype(np.float32)
    shift = np.random.randn(1,dim).astype(np.float32)*3
    Xo = np.vstack([(np.random.randn(n_out//2,dim)@L.T+shift).astype(np.float32),
                    (np.random.randn(n_out-n_out//2,dim)@L.T*2.5).astype(np.float32)])
    savemat(path, {"X":np.vstack([Xi,Xo]), "y":np.array([0]*n_in+[1]*n_out,dtype=np.float32).reshape(-1,1)})

def load_mat(path):
    mat=loadmat(path); X=mat["X"].astype(np.float32); y=mat["y"].astype(int).ravel()
    return X[y==0], X[y==1]

def get_odds(name):
    path=f"data/{name}.mat"
    if os.path.exists(path): src="real"
    elif try_pyod(name,path) or try_url(name,path): src="real"
    else: simulate(name,path); src="simulated"
    Xi,Xo = load_mat(path)
    return Xi,Xo,src

def get_kdd(rev=False, subset=50000):
    from sklearn.datasets import fetch_kddcup99
    data=fetch_kddcup99(subset="SA",percent10=True)
    X_raw=data.data; y_raw=(data.target==b"normal.").astype(int)
    cat=[1,2,3]; num=[i for i in range(X_raw.shape[1]) if i not in cat]
    enc=OneHotEncoder(sparse_output=False,handle_unknown="ignore")
    X=np.hstack([X_raw[:,num].astype(np.float32),enc.fit_transform(X_raw[:,cat])]).astype(np.float32)
    Xi=X[y_raw==(0 if rev else 1)]; Xo=X[y_raw==(1 if rev else 0)]
    rng=np.random.RandomState(42)
    ni=min(len(Xi),subset//2); no=min(len(Xo),subset//2)
    return Xi[rng.choice(len(Xi),ni,replace=False)],Xo[rng.choice(len(Xo),no,replace=False)],"sklearn"

print("Loading datasets...")
Xi_arr,Xo_arr,arr_src = get_odds("arrhythmia")
print(f"  Arrhythmia: n_in={len(Xi_arr)} n_out={len(Xo_arr)} dim={Xi_arr.shape[1]} src={arr_src}")
Xi_thy,Xo_thy,thy_src = get_odds("thyroid")
print(f"  Thyroid:    n_in={len(Xi_thy)} n_out={len(Xo_thy)} dim={Xi_thy.shape[1]} src={thy_src}")
try:
    Xi_kdd,Xo_kdd,_ = get_kdd(); kdd_ok=True
    print(f"  KDD:        n_in={len(Xi_kdd)} n_out={len(Xo_kdd)} dim={Xi_kdd.shape[1]}")
except: kdd_ok=False; print("  KDD: failed")
try:
    Xi_rev,Xo_rev,_ = get_kdd(rev=True); kddrev_ok=True
    print(f"  KDDRev:     n_in={len(Xi_rev)} n_out={len(Xo_rev)} dim={Xi_rev.shape[1]}")
except: kddrev_ok=False; print("  KDDRev: failed")


Loading datasets...
  Arrhythmia: n_in=386 n_out=66 dim=274 src=real
  Thyroid:    n_in=7018 n_out=182 dim=6 src=real
  KDD:        n_in=25000 n_out=3377 dim=99
  KDDRev:     n_in=3377 n_out=25000 dim=99


## Cell 3 — Model definitions + training utilities

In [3]:
class StudentTCDF(torch.autograd.Function):
    NU=5.0
    @staticmethod
    def forward(ctx,x):
        nu=StudentTCDF.NU; xn=x.detach().cpu().numpy()
        z=nu/(nu+np.clip(xn**2,1e-30,None))
        ib=betainc(nu/2,0.5,np.clip(z,1e-12,1-1e-12))
        ctx.save_for_backward(x)
        return torch.tensor(np.where(xn>=0,1.-ib/2.,ib/2.),dtype=x.dtype,device=x.device)
    @staticmethod
    def backward(ctx,g):
        x,=ctx.saved_tensors; nu=StudentTCDF.NU; xn=x.detach().cpu().numpy()
        lc=gammaln((nu+1)/2)-gammaln(nu/2)-.5*np.log(nu*np.pi)
        pdf=np.exp(lc-(nu+1)/2*np.log(1+xn**2/nu))
        return g*torch.tensor(pdf,dtype=x.dtype,device=x.device)
tcdf=StudentTCDF.apply

class BaseAE(nn.Module):
    def __init__(self,dim,hid,lat):
        super().__init__()
        self.W1=nn.Linear(dim,hid); self.W2=nn.Linear(hid,lat); self.dec=nn.Linear(lat,dim)
        for m in [self.W1,self.W2,self.dec]: nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def encode_elu(self,x): return self.W2(F.elu(F.silu(self.W1(x))))
    def forward(self,x):    return self.dec(self.encode_elu(x))
    def recon_err(self,x):
        with torch.no_grad(): return (x-self(x)).abs().mean(1)

class MELUGate(nn.Module):
    def __init__(self,lat):
        super().__init__()
        self.register_buffer("mu",torch.zeros(lat))
        self.register_buffer("Li",torch.eye(lat))
        self.register_buffer("tau",torch.tensor(1.5))
    def set_mcd(self,mu_np,Li_np,tv):
        dev=self.mu.device
        self.mu=torch.tensor(mu_np,dtype=torch.float32,device=dev)
        self.Li=torch.tensor(Li_np,dtype=torch.float32,device=dev)
        self.tau=torch.tensor(float(tv),device=dev)
    def mahal(self,Z):
        w=(Z-self.mu.unsqueeze(0))@self.Li.T; return w.norm(dim=1)

class MELU_AE(nn.Module):
    def __init__(self,dim,hid,lat):
        super().__init__()
        self.W1=nn.Linear(dim,hid); self.W2=nn.Linear(hid,lat); self.W3=nn.Linear(lat,dim)
        self.log_alpha=nn.Parameter(torch.log(torch.tensor(1.0)))
        self.log_beta =nn.Parameter(torch.log(torch.tensor(0.5)))
        self.gate=MELUGate(lat); self.gate_on=False
        for m in [self.W1,self.W2,self.W3]: nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    @property
    def alpha(self): return self.log_alpha.exp()
    @property
    def beta(self):  return self.log_beta.exp()
    def encode(self,x,ef=None):
        h1=F.silu(self.W1(x)); T1=h1*tcdf(h1)
        if self.gate_on and ef is not None:
            with torch.no_grad(): Zf=ef.encode_elu(x)
            m=self.gate.mahal(Zf); g=(m>=self.gate.tau).float().unsqueeze(1)
            amp=self.alpha*h1.sign()*torch.tanh(self.beta*(m.unsqueeze(1)-self.gate.tau).clamp(-8,8))
            return self.W2(T1+g*amp)
        return self.W2(T1)
    def forward(self,x,ef=None): return self.W3(self.encode(x,ef))
    def recon_err(self,x,ef=None):
        with torch.no_grad(): return (x-self(x,ef)).abs().mean(1)

def fast_mcd(Z,hf=0.75,ns=6,ni=5):
    n,d=Z.shape; h=max(int(n*hf),d+1); bd=np.inf; bm=bc=None
    for _ in range(ns):
        idx=np.random.choice(n,h,replace=False); sub=Z[idx]
        for _ in range(ni):
            mu=sub.mean(0); dv=sub-mu
            cov=dv.T@dv/max(len(sub)-1,1)+1e-4*np.eye(d)
            Si=np.linalg.inv(cov)
            ds=np.sqrt(np.maximum(np.einsum("bi,ij,bj->b",Z-mu,Si,Z-mu),0))
            idx=np.argsort(ds)[:h]; sub=Z[idx]
        mu=sub.mean(0); dv=sub-mu; cov=dv.T@dv/max(len(sub)-1,1)
        det=np.linalg.det(cov+1e-4*np.eye(d))
        if det<bd: bd=det; bm=mu; bc=cov
    try:
        L=np.linalg.cholesky(bc+1e-4*np.eye(d)); Li=np.linalg.inv(L)
        if np.isnan(Li).any() or np.linalg.cond(Li)>1e7: Li=np.eye(d)
    except: Li=np.eye(d)
    return bm,bc,Li

def train_melu(Xi_tr,Xi_all_sc,seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=Xi_tr.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr_t=torch.tensor(Xi_tr,dtype=torch.float32)
    ae=BaseAE(dim,hid,lat); opt1=optim.Adam(ae.parameters(),lr=LR); wu1=int(EPOCHS_PRE*.2)
    for ep in range(EPOCHS_PRE):
        ae.train(); perm=torch.randperm(len(Xi_tr_t))
        for i in range(0,len(Xi_tr_t),64):
            xb=Xi_tr_t[perm[i:i+64]]; er=(xb-ae(xb)).abs().mean(1); loss=er.mean()
            if ep>=wu1:
                ae.eval()
                with torch.no_grad(): er_all=ae.recon_err(Xi_tr_t).numpy()
                py=torch.tensor((er_all>np.percentile(er_all,PCT_PSEUDO)).astype(np.float32))
                ae.train()
                em,eM=er.detach().min(),er.detach().max()
                pb=((er-em)/(eM-em+1e-8)).clamp(1e-6,1-1e-6)
                loss=loss+LAM_BCE*F.binary_cross_entropy(pb,py[perm[i:i+64]])
            opt1.zero_grad(); loss.backward(); opt1.step()
    ae.eval()
    with torch.no_grad(): Z_tr=ae.encode_elu(Xi_tr_t).numpy()
    mu_l,_,Li_l=fast_mcd(Z_tr); w=(Z_tr-mu_l)@Li_l.T
    dm=np.sqrt(np.maximum((w**2).sum(1),0)); tau=float(np.percentile(dm,TAU_PERCENTILE))
    melu=MELU_AE(dim,hid,lat)
    melu.W1.weight.data=ae.W1.weight.data.clone(); melu.W1.bias.data=ae.W1.bias.data.clone()
    melu.W2.weight.data=ae.W2.weight.data.clone(); melu.W2.bias.data=ae.W2.bias.data.clone()
    melu.W3.weight.data=ae.dec.weight.data.clone(); melu.W3.bias.data=ae.dec.bias.data.clone()
    melu.gate.set_mcd(mu_l,Li_l,tau)
    for p in ae.parameters(): p.requires_grad_(False)
    opt3=optim.Adam(melu.parameters(),lr=LR*.5); wu3=int(EPOCHS_FINE*.2)
    for ep in range(EPOCHS_FINE):
        melu.gate_on=(ep>=wu3); melu.train(); perm=torch.randperm(len(Xi_tr_t))
        for i in range(0,len(Xi_tr_t),64):
            xb=Xi_tr_t[perm[i:i+64]]; er=(xb-melu(xb,ae)).abs().mean(1); loss=er.mean()
            if ep>=wu3:
                melu.eval()
                with torch.no_grad(): er_all=melu.recon_err(Xi_tr_t,ae).numpy()
                py=torch.tensor((er_all>np.percentile(er_all,PCT_PSEUDO)).astype(np.float32))
                melu.train()
                em,eM=er.detach().min(),er.detach().max()
                pb=((er-em)/(eM-em+1e-8)).clamp(1e-6,1-1e-6)
                loss=loss+LAM_BCE*F.binary_cross_entropy(pb,py[perm[i:i+64]])
            opt3.zero_grad(); loss.backward(); opt3.step()
    melu.gate_on=True; melu.eval()
    return ae,melu,float((dm>tau).mean())

def train_elu(Xi_tr,seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    dim=Xi_tr.shape[1]; lat=lat_for(dim); hid=hid_for(dim)
    Xi_tr_t=torch.tensor(Xi_tr,dtype=torch.float32)
    ae=BaseAE(dim,hid,lat); opt=optim.Adam(ae.parameters(),lr=LR); wu=int((EPOCHS_PRE+EPOCHS_FINE)*.2)
    n_ep=EPOCHS_PRE+EPOCHS_FINE
    for ep in range(n_ep):
        ae.train(); perm=torch.randperm(len(Xi_tr_t))
        for i in range(0,len(Xi_tr_t),64):
            xb=Xi_tr_t[perm[i:i+64]]; er=(xb-ae(xb)).abs().mean(1); loss=er.mean()
            if ep>=wu:
                ae.eval()
                with torch.no_grad(): er_all=ae.recon_err(Xi_tr_t).numpy()
                py=torch.tensor((er_all>np.percentile(er_all,PCT_PSEUDO)).astype(np.float32))
                ae.train()
                em,eM=er.detach().min(),er.detach().max()
                pb=((er-em)/(eM-em+1e-8)).clamp(1e-6,1-1e-6)
                loss=loss+LAM_BCE*F.binary_cross_entropy(pb,py[perm[i:i+64]])
            opt.zero_grad(); loss.backward(); opt.step()
    ae.eval(); return ae

def score_and_eval(scores_np, y_test, n_anom):
    auroc=float(roc_auc_score(y_test,scores_np))
    thresh=np.sort(scores_np)[::-1][n_anom-1]
    yp=(scores_np>=thresh).astype(int)
    f1=float(f1_score(y_test,yp,zero_division=0))
    return auroc,f1

print("All model utilities defined.")


All model utilities defined.


## Cell 4 — Run all 5 diagnostic tests

> **Runtime:** ~15–30 min per dataset (dominated by MELU + ELU multi-run training)
>
> Each dataset runs:
> - N=20 runs of MELU-Δt + ELU-AE (Zong protocol)
> - 1 run of IForest / LOF / OCSVM
> - 1 run of Random AE
> - 1 run of Supervised LR upper bound


In [ ]:
DATASETS = {}
sc_dict  = {}
if "Xi_arr" in dir():
    sc=StandardScaler().fit(Xi_arr)
    DATASETS["Arrhythmia"] = (sc.transform(Xi_arr), sc.transform(Xo_arr), arr_src)
    sc_dict["Arrhythmia"]=sc
if "Xi_thy" in dir():
    sc=StandardScaler().fit(Xi_thy)
    DATASETS["Thyroid"] = (sc.transform(Xi_thy), sc.transform(Xo_thy), thy_src)
    sc_dict["Thyroid"]=sc
if "kdd_ok" in dir() and kdd_ok:
    sc=StandardScaler().fit(Xi_kdd)
    DATASETS["KDD"] = (sc.transform(Xi_kdd), sc.transform(Xo_kdd), "sklearn")
    sc_dict["KDD"]=sc
if "kddrev_ok" in dir() and kddrev_ok:
    sc=StandardScaler().fit(Xi_rev)
    DATASETS["KDDRev"] = (sc.transform(Xi_rev), sc.transform(Xo_rev), "sklearn")
    sc_dict["KDDRev"]=sc

# Results store: {ds: {method: [auroc, f1, ...]}}
ALL = {ds: {} for ds in DATASETS}

for ds_name,(Xi_sc,Xo_sc,src) in DATASETS.items():
    dim = Xi_sc.shape[1]; n_in=len(Xi_sc); n_out=len(Xo_sc)
    print(f"\n{'='*58}")
    print(f"{ds_name}  dim={dim}  n_in={n_in}  n_out={n_out}  src={src}")
    print(f"{'='*58}")

    melu_f1s=[]; melu_aus=[]; elu_f1s=[]; elu_aus=[]
    rand_aus=[]; scores_melu_last=None; scores_elu_last=None
    y_test_last=None; Xi_te_last=None

    n_runs = 20 if ds_name in ["Arrhythmia","Thyroid"] else 10
    t0=time.time()

    for seed in range(n_runs):
        rng=np.random.RandomState(seed); idx=rng.permutation(n_in)
        n_tr=max(8,int(n_in*TRAIN_FRAC))
        Xi_tr=Xi_sc[idx[:n_tr]]; Xi_te=Xi_sc[idx[n_tr:]]
        X_test=np.vstack([Xi_te,Xo_sc])
        y_test=np.array([0]*len(Xi_te)+[1]*n_out,dtype=np.float32)
        n_anom=n_out; X_te_t=torch.tensor(X_test,dtype=torch.float32)

        # ── TEST 1: MELU-Δt ──────────────────────────────────────────────────
        ae,melu,gp = train_melu(Xi_tr,Xi_sc,seed)
        sc_m=melu.recon_err(X_te_t,ae).numpy(); au_m,f1_m=score_and_eval(sc_m,y_test,n_anom)
        melu_aus.append(au_m); melu_f1s.append(f1_m)

        # ── TEST 2: ELU-AE (critical baseline) ───────────────────────────────
        elu_ae=train_elu(Xi_tr,seed)
        sc_e=elu_ae.recon_err(X_te_t).numpy(); au_e,f1_e=score_and_eval(sc_e,y_test,n_anom)
        elu_aus.append(au_e); elu_f1s.append(f1_e)

        # ── TEST 3: Random AE (no training) ──────────────────────────────────
        torch.manual_seed(seed); rand_ae=BaseAE(dim,hid_for(dim),lat_for(dim))
        sc_r=rand_ae.recon_err(X_te_t).numpy()
        au_r=float(roc_auc_score(y_test,sc_r)); rand_aus.append(au_r)

        # Save last seed for plots
        if seed==0:
            scores_melu_last=sc_m; scores_elu_last=sc_e; y_test_last=y_test; Xi_te_last=Xi_te

    # ── TEST 4 (once): Shallow baselines ─────────────────────────────────────
    rng=np.random.RandomState(0); idx=rng.permutation(n_in)
    n_tr=max(8,int(n_in*TRAIN_FRAC))
    Xi_tr0=Xi_sc[idx[:n_tr]]; Xi_te0=Xi_sc[idx[n_tr:]]
    X_t0=np.vstack([Xi_te0,Xo_sc]); y_t0=np.array([0]*len(Xi_te0)+[1]*n_out,dtype=np.float32)

    sh_results={}
    for name,clf in [
        ("IForest", IsolationForest(n_estimators=200,contamination=0.1,random_state=42)),
        ("LOF",     LocalOutlierFactor(n_neighbors=20,novelty=True,contamination=0.1)),
        ("OCSVM",   OneClassSVM(nu=0.1,kernel="rbf",gamma="auto")),
    ]:
        try:
            clf.fit(Xi_tr0)
            sc_sh=-clf.score_samples(X_t0) if hasattr(clf,"score_samples") else -clf.decision_function(X_t0)
            au_sh=float(roc_auc_score(y_t0,sc_sh))
            thresh=np.sort(sc_sh)[::-1][n_out-1]
            f1_sh=float(f1_score(y_t0,(sc_sh>=thresh).astype(int),zero_division=0))
            sh_results[name]=(au_sh,f1_sh)
        except Exception as e:
            sh_results[name]=(0.5,0.0)

    # ── TEST 5: Supervised upper bound ───────────────────────────────────────
    try:
        X_sup=np.vstack([Xi_tr0,Xo_sc])
        y_sup=np.array([0]*len(Xi_tr0)+[1]*n_out)
        lr=LogisticRegression(max_iter=1000,C=1.0).fit(X_sup,y_sup)
        prob=lr.predict_proba(X_t0)[:,1]
        au_sup=float(roc_auc_score(y_t0,prob))
        thresh_sup=np.sort(prob)[::-1][n_out-1]
        f1_sup=float(f1_score(y_t0,(prob>=thresh_sup).astype(int),zero_division=0))
    except Exception as e:
        au_sup,f1_sup=0.5,0.0

    ALL[ds_name] = {
        "MELU-Dt":  (np.mean(melu_aus),np.std(melu_aus),np.mean(melu_f1s),np.std(melu_f1s)),
        "ELU-AE":   (np.mean(elu_aus), np.std(elu_aus), np.mean(elu_f1s), np.std(elu_f1s)),
        "RandAE":   (np.mean(rand_aus),np.std(rand_aus),None,None),
        "IForest":  (sh_results["IForest"][0],0,sh_results["IForest"][1],0),
        "LOF":      (sh_results["LOF"][0],0,sh_results["LOF"][1],0),
        "OCSVM":    (sh_results["OCSVM"][0],0,sh_results["OCSVM"][1],0),
        "Supervised-LR": (au_sup,0,f1_sup,0),
        "_scores_melu": scores_melu_last,
        "_scores_elu":  scores_elu_last,
        "_y_test":  y_test_last,
        "_n_anom":  n_out,
    }

    # Print summary
    elapsed=time.time()-t0
    print(f"\n  Results [{elapsed:.0f}s]:")
    print(f"  {'Method':<16} {'AUROC':>8}  {'F1':>8}")
    print(f"  {'-'*36}")
    for m in ["MELU-Dt","ELU-AE","RandAE","IForest","LOF","OCSVM","Supervised-LR"]:
        au,aus,f1,f1s = ALL[ds_name][m]
        f1_str=f"{f1*100:.1f}%" if f1 is not None else "  N/A "
        print(f"  {m:<16} {au:.4f}   {f1_str}")

    delta_melu_elu = np.mean(melu_aus)-np.mean(elu_aus)
    print(f"\n  MELU-Dt vs ELU-AE delta AUROC: {delta_melu_elu:+.4f}")
    if abs(delta_melu_elu) < 0.01:
        print("  --> MELU-Dt ~ ELU-AE: result driven by reconstruction error, not activation")
    elif delta_melu_elu > 0.02:
        print("  --> MELU-Dt > ELU-AE: activation function genuinely improves detection")
    else:
        print("  --> Marginal difference; needs more runs to confirm")

    if sh_results["IForest"][0] > 0.95:
        print("  --> IForest AUROC > 0.95: dataset is trivially easy for ANY one-class method")
    if np.mean(rand_aus) > 0.70:
        print(f"  --> Random AE AUROC = {np.mean(rand_aus):.3f}: feature structure alone separates classes")
    if au_sup > 0.98:
        print(f"  --> Supervised LR AUROC = {au_sup:.3f}: classes ARE inherently linearly separable")

print("\nAll diagnostics complete.")



Arrhythmia  dim=274  n_in=386  n_out=66  src=real

  Results [893s]:
  Method              AUROC        F1
  ------------------------------------
  MELU-Dt          1.0000   99.8%
  ELU-AE           1.0000   100.0%
  RandAE           1.0000     N/A 
  IForest          1.0000   100.0%
  LOF              1.0000   100.0%
  OCSVM            1.0000   100.0%
  Supervised-LR    1.0000   100.0%

  MELU-Dt vs ELU-AE delta AUROC: -0.0000
  --> MELU-Dt ~ ELU-AE: result driven by reconstruction error, not activation
  --> IForest AUROC > 0.95: dataset is trivially easy for ANY one-class method
  --> Random AE AUROC = 1.000: feature structure alone separates classes
  --> Supervised LR AUROC = 1.000: classes ARE inherently linearly separable

Thyroid  dim=6  n_in=7018  n_out=182  src=real

  Results [9066s]:
  Method              AUROC        F1
  ------------------------------------
  MELU-Dt          0.9439   74.0%
  ELU-AE           0.9267   71.4%
  RandAE           0.9391     N/A 
  IForest   

## Cell 5 — Visualisation (AUROC bars + score distributions)

In [ ]:
avail = list(ALL.keys())
if not avail:
    print("Run Cell 4 first.")
else:
    n_ds = len(avail)
    fig  = plt.figure(figsize=(22, 5*n_ds))
    gs   = gridspec.GridSpec(n_ds, 2, figure=fig, hspace=0.55, wspace=0.35)
    fig.suptitle("Diagnostic verification — MELU-Dt vs all baselines + score distributions",
                 fontsize=13, fontweight="bold")

    for row, ds in enumerate(avail):
        d    = ALL[ds]
        methods  = ["MELU-Dt","ELU-AE","RandAE","IForest","LOF","OCSVM","Supervised-LR"]
        COLORS_D = {"MELU-Dt":"#1D9E75","ELU-AE":"#534AB7","RandAE":"#AAAAAA",
                    "IForest":"#BA7517","LOF":"#D85A30","OCSVM":"#888780",
                    "Supervised-LR":"#000000"}

        # ── Panel A: AUROC bars ───────────────────────────────────────────────
        ax = fig.add_subplot(gs[row, 0])
        aus   = [d[m][0] for m in methods]
        stds  = [d[m][1] for m in methods]
        bars  = ax.bar(methods, aus, color=[COLORS_D[m] for m in methods], alpha=0.85)
        ax.errorbar(range(len(methods)), aus, yerr=stds,
                    fmt="none", ecolor="black", capsize=3, lw=1)
        ax.axhline(0.5, color="gray", lw=0.8, ls="--", label="random (0.5)")
        ax.axhline(0.95, color="#D85A30", lw=1, ls=":", alpha=0.6, label="0.95 threshold")
        ax.set_ylim(0.3, 1.05)
        ax.set_ylabel("AUROC", fontsize=11)
        ax.set_title(f"{ds} — AUROC all methods", fontsize=11)
        ax.tick_params(axis="x", rotation=40, labelsize=9)
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.25)
        for b, v in zip(bars, aus):
            ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.3f}",
                    ha="center", fontsize=8, fontweight="bold")

        # ── Panel B: Score distributions ──────────────────────────────────────
        ax = fig.add_subplot(gs[row, 1])
        sm = d.get("_scores_melu"); se = d.get("_scores_elu"); yt = d.get("_y_test")
        if sm is not None and yt is not None:
            in_m  = sm[yt == 0]; out_m = sm[yt == 1]
            in_e  = se[yt == 0]; out_e = se[yt == 1]
            kw = dict(alpha=0.55, density=True, bins=40)
            ax.hist(in_m,  color="#1D9E75", label="MELU inliers",  **kw)
            ax.hist(out_m, color="#085041", label="MELU outliers", histtype="step",
                    linewidth=2, **kw)
            ax.hist(in_e,  color="#534AB7", label="ELU inliers",  **kw)
            ax.hist(out_e, color="#1D2A7A", label="ELU outliers", histtype="step",
                    linewidth=2, linestyle="--", **kw)
            ax.set_xlabel("Reconstruction error", fontsize=10)
            ax.set_ylabel("Density", fontsize=10)
            ax.set_title(f"{ds} — Score distributions (seed=0)", fontsize=11)
            ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.25)

            overlap = (np.min(out_m) < np.max(in_m))
            ax.text(0.98, 0.95,
                    "Overlap: YES" if overlap else "Overlap: NO (perfect sep.)",
                    transform=ax.transAxes, ha="right", va="top", fontsize=9,
                    color="#D85A30" if overlap else "#1D9E75",
                    bbox=dict(boxstyle="round,pad=0.3",
                              fc="white" if overlap else "#E1F5EE", alpha=0.8))

    plt.savefig("outputs/nb17_verification.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: outputs/nb17_verification.png")


## Cell 6 — Verdict summary

In [ ]:
print("=" * 70)
print("VERIFICATION SUMMARY")
print("=" * 70)
print()
print(f"{'Dataset':<14} {'MELU-Dt':>10} {'ELU-AE':>10} {'Δ(M-E)':>9} "
      f"{'IForest':>9} {'RandAE':>9} {'Sup-LR':>9}  Verdict")
print("-" * 90)

PUBLISHED_BEST = {
    "Arrhythmia": 67.2,
    "Thyroid":    93.7,
    "KDD":        99.9,
    "KDDRev":     98.9,
}

for ds in ALL:
    d   = ALL[ds]
    mau = d["MELU-Dt"][0]; eau = d["ELU-AE"][0]
    rau = d["RandAE"][0];  iau = d["IForest"][0]
    sau = d["Supervised-LR"][0]
    dm  = mau - eau

    # Verdict logic
    if rau > 0.80:
        verdict = "TRIVIAL: random features separate classes"
    elif iau > 0.95:
        verdict = "EASY: IForest also near-perfect"
    elif dm < 0.01:
        verdict = "AE-DRIVEN: MELU ~ ELU, not activation-specific"
    elif dm > 0.03:
        verdict = "GENUINE: MELU meaningfully > ELU"
    else:
        verdict = "MARGINAL: small gain, needs more runs"

    print(f"  {ds:<14} {mau:>9.4f} {eau:>9.4f} {dm:>+8.4f} "
          f"{iau:>8.4f} {rau:>8.4f} {sau:>8.4f}  {verdict}")

print()
print("Interpretations:")
print("  TRIVIAL     → any method wins; not useful for paper")
print("  EASY        → dataset not suitable for benchmark comparison")
print("  AE-DRIVEN   → reconstruction error beats DAGMM/GOAD, but MELU = ELU")
print("  GENUINE     → MELU-Δt activation function genuinely improves detection")
print("  MARGINAL    → weak signal; avoid claiming this dataset as a win")
print()

# Compare against published
print("vs published baselines (best published F1, ours converted via AUROC):")
for ds in ALL:
    if ds in PUBLISHED_BEST:
        mf1 = ALL[ds]["MELU-Dt"][2]*100 if ALL[ds]["MELU-Dt"][2] else 0
        ef1 = ALL[ds]["ELU-AE"][2]*100  if ALL[ds]["ELU-AE"][2]  else 0
        pub = PUBLISHED_BEST[ds]
        print(f"  {ds:<14}  Published best={pub:.1f}%  "
              f"MELU-Dt={mf1:.1f}%  ELU-AE={ef1:.1f}%")


## Cell 7 — Export CSV

In [ ]:
rows = []
for ds,d in ALL.items():
    for method in ["MELU-Dt","ELU-AE","RandAE","IForest","LOF","OCSVM","Supervised-LR"]:
        if method.startswith("_"): continue
        au,aus,f1,f1s = d[method]
        rows.append({"dataset":ds,"method":method,
                     "auroc_mean":round(au,4),"auroc_std":round(aus,4),
                     "f1_mean":round(f1,4) if f1 is not None else None,
                     "f1_std":round(f1s,4) if f1s is not None else None})
pd.DataFrame(rows).to_csv("outputs/nb17_verification.csv",index=False)
print("Saved: outputs/nb17_verification.csv")
